# Лабораторная работа 5. Создание чатбота на основе LLM

#### Работу выполнил: {впишите вашу фамилию} {впишите ваше имя}, студент группы ФИб-4

**Для повышения скорости работы подключите GPU в меню `Среда выполнения`. LLM и её входные данные загружайте на видеокарту.**

# Чать 1. Знакомство с библиотекой `transformers`

Классы `AutoTokenizer` и `AutoModelForCausalLM` позволяет загрузить чекпоинты языковой модели и выполнить генерацию текста.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio

В качестве модели для экспериментов возьмите модель `Qwen/Qwen3-0.6B` с huggingface. Познакомьтесь с описанием модели и её использованием [ссылка](https://huggingface.co/Qwen/Qwen3-0.6B). Веса загружайте в типе `torch.bfloat16` для экономии памяти GPU. Не забудьте в интерфейсе colab подключить GPU.

In [ ]:
model_name_or_path = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name_or_path, dtype=torch.bfloat16, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

Входной текст для генерации может быть представлен в режиме диалога в виде списка объектов с полями `role` и `content`. Значения поля `role` может принимать значения `system`, `user`, `assistent`, что соответсвует системному, пользовательскому промптам и ответу модели. У каждой модели специальные токены и собственный формат, приведение к которому происходит с помощью метода `apply_chat_template()`

In [ ]:
chat = [
  {"role": "system", "content": "Отвечай на русском языке"},
  {"role": "user", "content": "Что ты думаешь о законах робототехники?"},
  {"role": "assistant", "content": "Законы робототехники — это нарастающая область правовой регуляции, которая стремится адаптироваться к быстрому развитию технологий в области искусственного интеллекта (ИИ) и автоматизации."},
  {"role": "user", "content": "А какой второй закон?"}
]
text = tokenizer.apply_chat_template(chat, tokenize=False,
                                       add_generation_prompt=True,
                                       return_tensors="pt"
)
print(text)

Генерация выполняется с помощью метода generate(). Генерируемый текст большой (с рассуждениями), поэтому раскройте поле вывода.

In [ ]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)

In [ ]:
all_generated_text = tokenizer.decode(generated_ids[0])
print(all_generated_text)

Изучите формат ответа модели и распарсите ответ, например следующим образом.

In [ ]:
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
print(f"Рассуждения: {thinking_content}")
print(f"Ответ: {content}")

**Задание 1.** Напишите чатбот c поддержкой истории при генерации ответа. Для ввода сообщений пользователя используйте функцию `input()`, в случае введения пользователем текста `выход`, диалог должен останавливаться, а исполнение кода прекращаться.

Глубина истории должна задаваться параметром.

Сделайте логирование входных данных модели в текстовый файл и проверьте правильность отправляемых данных в модель для генерации.

Поддержку истории проверяйте следующим диалогом при установленной длине истории 1 (одно предыдущее сообщение) или при большей длине, но тогда адаптируйте диалог:
```
USER: Какой первый закон робототехники?
ASSISTANT: ....
USER: А какой второй закон?
ASSISTANT:
```

In [ ]:
#ваш код

# Часть 2. Знакомство с `langchain`. Простой пример RAG

В colab-сессии скорее всего уже установлены требуемые модули, но если нет, то скорее всего потребуются следующие.

In [ ]:
# !pip install -U langchain langchain-core
# !pip install -U langchain-community langchain-text-splitters
# !pip install -U langchain-huggingface sentence-transformers

Выполним импорт требуемых модулей.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_classic.document_loaders import DirectoryLoader, JSONLoader

Также установим дополнительные модули.

In [ ]:
!pip install jq
!pip install chromadb

Определим переменные для каталога с документами, по которым будет выполняться поиск, и для каталога под векторную базу данных.

In [ ]:
data_directory = "/content/documents"
db_directory = "/content/chroma_db"
load_new_db = False

Загрузим `csv`-файл новостного датасета, который использовали ранее заданиях, и пересохраним данные в `json`-файл.

In [ ]:
import pandas as pd
df = pd.read_csv('lenta_ru_news_filtered.csv')
df.to_json('lenta_ru_news_filtered.json', orient='records', force_ascii=False, indent=4)

Переместите полученный `json`-файл с текстами в `data_directory`.

Инициализируем модель, с помощью которой будут строиться эмбеддинги текстовых фагментов документов.

In [ ]:
data_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Загрузим документы. Пусть документы представляют собой `json` файлы со списками объектов полем `text`, в котором хранится текст (пример файла находится в гугл-папке вместе с блокнотом лабораторной работы).

In [ ]:
loader = DirectoryLoader(data_directory, glob="**/*.json", show_progress=True, loader_cls=JSONLoader, loader_kwargs={"jq_schema":".[].text"})
documents = loader.load()
print(f"Count of documents is {len(documents)}")

Разделим тексты на фрагменты.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=10)
texts = text_splitter.split_documents(documents)
print(f"Count of hunks is {len(texts)}")

Наполним векторную базу данных.

In [ ]:
vector_db = Chroma.from_documents(documents=texts, embedding=data_embeddings, persist_directory=db_directory)

Создадим объект, через который будет выполняться поиск.\
`search_type="mmr"` - это метод MMR (**Maximal Marginal Relevance** — максимальная маржинальная релевантность). В отличие от стандартного поиска по сходству, который просто выдает самые похожие документы, MMR старается найти баланс между релевантностью запросу и разнообразием ответов.

In [ ]:
retriever = vector_db.as_retriever(search_type="mmr")

Зададим запрос и выполним поиск.

In [ ]:
query = "Самая высокооплачиваемая писательница"
response = retriever.invoke(query)
results = [{"id":i, "text":x.page_content, "source":x.metadata['source']} for i,x in enumerate(response)]
for x in results:
  print(x)

**Задание 2.** Реализуйте поиск по новостным текстам из датасета предыдущих лабораторных работ. Протестируйте работу поиска. Сравните реализованный поиск с поиском на основе `TF-IDF` из предыдущих лабораторных работ, используя прежние запросы. Сделайте выводы.

In [ ]:
# ваш код

# Часть 3. Чатбот с использованием внешних знаний

**Задание 3.** Напишите чатбот, в котором перед генерацией выполнятся поиск внешних знаний, а затем выполняется генерация ответа пользователю с учётом этих знаний и истории диалога (тестируйте на глубине истории не более 5 сообщений диалога). Придумайте что передавать в качестве запроса в векторную БД (например, можно попросить LLM суммаризовать историю диалога или самой сформулировать запрос из 5-10 слов) - попробуйте несколько вариантов и выберите более эффективный.

Протестируйте работу получившегося чатбота. Сделайте выводы.

In [ ]:
# ваш код